# Setup

In [ ]:
DATA_SOURCES = ["buysite"]

## Pneuma-Seeker-Specific

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
import time

from torch.backends import cudnn

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

sys.path.append("../..")

from pneuma_seeker.core.conductor.chat_interface import ChatInterface
from pneuma_seeker.core.ir_system.data_model import convert_multi_retriever_results_to_str
from pneuma_seeker.model.llm_message import LLMMessage, Role
from pneuma_seeker.core.conductor.state import InformationNeedState
from pneuma_seeker.core.ir_system.data_model import AbstractDocument, RetrieverType

In [ ]:
llm_path = "o4-mini"
embed_model_path = "../model/weight/bge-base"
chat_interface = ChatInterface(
    llm_path,
    embed_model_path,
    "user_id",
    "chat_id",
    DATA_SOURCES,
    False,
    "../../../.env"
)

## Scenario-Specific

In [ ]:
def get_formatted_system_output(
    system_output: str,
    state: InformationNeedState,
    curr_retrieval_results: dict[RetrieverType, list[AbstractDocument]],
):
    return f"""SYSTEM OUTPUT:
```{system_output}```

STATE:
```{state}```

RETRIEVED DATA BY THE SYSTEM:
```{convert_multi_retriever_results_to_str(curr_retrieval_results)}```
"""

In [ ]:
class ScenarioTester:
    def __init__(self) -> None:
        self.chat_messages: list[LLMMessage] = []
    
    def chat(self, prompt: str, external_data_paths: list[str] = []):
        start = time.time()

        self.chat_messages.append(
            LLMMessage(role=Role.USER.value, content=prompt)
        )

        system_output = ""
        for log_message in chat_interface.process_user_input(
            self.chat_messages, external_data_paths
        ):
            if log_message.startswith("LOG") or log_message.startswith("DONE"):
                continue
            system_output += log_message
        end = time.time()
        print(f"==> Responding in {end-start:.2f} seconds: {system_output}")

        self.chat_messages.append(
            LLMMessage(
                role=Role.ASSISTANT.value,
                content=system_output,
            )
        )

        formatted_sys_output = get_formatted_system_output(
            system_output,
            chat_interface.llm_conductor.info_need_state,
            chat_interface.llm_conductor.current_retrieval_results,
        )
        print(f"==> Response:\n{formatted_sys_output}")

# Interaction

In [ ]:
scenario_tester = ScenarioTester()

In [ ]:
scenario_tester.chat("How are you?", [])